In [1]:
import cv2
import numpy as np
import os
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, precision_score, recall_score, f1_score
from sklearn.metrics.pairwise import rbf_kernel

# Define the relu_kernel function before using it in param_grid
def relu_kernel(X, Y=None, gamma=None):
    if Y is None:
        Y = X
    K = rbf_kernel(X, Y, gamma=gamma)
    K = np.maximum(K, 0)
    return K

# Load the dataset (assuming it's a folder with real and fake images)
def load_dataset(data_path):
    images = []
    labels = []
    for folder in ['real', 'fake']:
        folder_path = os.path.join(data_path, folder)
        for file in os.listdir(folder_path):
            img_path = os.path.join(folder_path, file)
            img = cv2.imread(img_path)
            img = cv2.resize(img, (64, 64))  # Resize to a fixed size
            images.append(img)
            labels.append(folder)
    return np.array(images), np.array(labels)

# Encode labels as integers
def encode_labels(labels):
    label_encoder = LabelEncoder()
    return label_encoder.fit_transform(labels)

# Main function to train and evaluate the model
def main():
    data_path = 'C://Users//Sinchan A//Desktop//Internship//vid'
    images, labels = load_dataset(data_path)
    labels = encode_labels(labels)

    # Split the dataset into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(images, labels, test_size=0.3, random_state=42)

    # Flatten the images
    X_train = X_train.reshape(X_train.shape[0], -1)
    X_test = X_test.reshape(X_test.shape[0], -1)

    # Convert data types if needed
    X_train = X_train.astype(np.float32)
    X_test = X_test.astype(np.float32)
    y_train = y_train.astype(np.int32)
    y_test = y_test.astype(np.int32)

    # Define the parameter grid for grid search
    param_grid = {
        'C': [0.1, 1, 10, 100],  # Regularization parameter
        'gamma': [0.001, 0.01, 0.1],  # Kernel coefficient
        'kernel': ['linear', 'rbf', 'poly', 'sigmoid']  # Kernel functions
    }

    # Create the SVM classifier
    svm = SVC()

    # Perform grid search with cross-validation
    grid_search = GridSearchCV(estimator=svm, param_grid=param_grid, cv=5, scoring='accuracy', n_jobs=-1)
    grid_search.fit(X_train, y_train)

    # Get the best hyperparameters
    best_params = grid_search.best_params_
    print(f"Best parameters: {best_params}")

    # Get the best estimator
    best_model = grid_search.best_estimator_

    # Make predictions on the test set
    y_pred = best_model.predict(X_test)

    # Evaluate the model
    accuracy = accuracy_score(y_test, y_pred)
    print(f'Accuracy: {accuracy}')
    print(classification_report(y_test, y_pred))

    # Calculate precision, recall, and F1-score
    precision = precision_score(y_test, y_pred)
    print(f"Precision: {precision}")

    recall = recall_score(y_test, y_pred)   
    print(f"Recall: {recall}")

    f1 = f1_score(y_test, y_pred)
    print(f"F1-score: {f1}")

if __name__ == "__main__":
    main()


Best parameters: {'C': 0.1, 'gamma': 0.001, 'kernel': 'linear'}
Accuracy: 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1287
           1       1.00      1.00      1.00      1399

    accuracy                           1.00      2686
   macro avg       1.00      1.00      1.00      2686
weighted avg       1.00      1.00      1.00      2686

Precision: 1.0
Recall: 1.0
F1-score: 1.0
